# Table 1 — datasets (train + test)

**🟡 moderate** · source: `notebooks/dataset_split_counts.py`

Reads the training-atlas obs (backed) + split caches.

## Configuration — edit the paths, then run

In [ ]:
import os, sys, glob, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
warnings.filterwarnings("ignore")
try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print("[note] scanpy/anndata not available:", e)

# ── EDIT THESE PATHS to match your environment ──
REPO_ROOT     = Path("/path/to/spatnic")          # this repository
BACKUP_ROOT   = Path("/path/to/backup")           # integrate_adata_filtered.h5ad, galaxy scores, Liver meta
DATA_ROOT     = Path("/path/to/data")             # GxD concat, lung annotated, c2l refs, spatnic_models, GxD_Xenium
BENCHMARK_DB  = Path("/path/to/benchmark_db")     # Xenium/VisiumHD/MERFISH/CosMx + adata_hvg_*
VISIUMHD_ROOT = Path("/path/to/VisiumHD")         # Visium HD ADC track
WEIGHTS_DIR   = Path.home() / ".spatnic" / "weights"

# ── Derived ──
NB     = REPO_ROOT / "notebooks"
COMP   = NB / "comparison_results"
VHD    = VISIUMHD_ROOT
MODELS = DATA_ROOT / "spatnic_models"
PAPER  = REPO_ROOT / "paper"
BASE_DIR  = VHD          # VisiumHD ADC notebook global
THRESHOLD = 0.9          # overridden to 0.5 by the Fig 5 shortcut setup cell
sys.path[:0] = [str(REPO_ROOT / "scripts"), str(NB)]
if NB.exists():
    os.chdir(NB)         # extracted cells were written for cwd = notebooks/

def _tbl(csv, n=None):
    p = Path(csv)
    if not p.exists():
        print("[missing]", p); return None
    df = pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p)
    display(df.head(n) if n else df); return df

def _run(script, show=None, n=None):
    import subprocess
    cmd = f"python notebooks/{script}"
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=str(REPO_ROOT), capture_output=True, text=True)
    print((r.stdout or "")[-3000:])
    if r.returncode: print("STDERR:\n", (r.stderr or "")[-2000:])
    if show: _tbl(COMP / show, n)


## Regenerate (runs the real metric program)

In [ ]:
_run("dataset_split_counts.py", show="eval_confmat/dataset_split_counts.csv")

## Result (current cached values)

**Datasets** (`dataset_split_counts.csv`, 4 rows)

| dataset | train_samples | train_cells | train_tumor | train_normal | test_samples | test_cells | test_tumor | test_normal | note |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Primary (internal CRC Xenium) | 52 | 391214 | 195607 | 195607 | 52 | 97804 | 48902 | 48902 | cell-level split; same patients in train |
| Primary external (GxD primary CRC) | 0 | 0 | 0 | 0 | 7 | 673701 | 570856 | 102845 | external test only (never used for train |
| Liver met (GxD) | 12 | 200000 | 100000 | 100000 | 3 | 273756 | 193375 | 80381 | sample-level 12/3 split (seed42); train  |
| Lung met (GxD) | 4 | 151998 | 75999 | 75999 | 1 | 55076 | 45495 | 9581 | sample-level 4/1 split; train balanced;  |